In [4]:
import sys
sys.path.append("..")

import torch
import pandas as pd
from torch.utils.data import DataLoader, random_split
from sklearn.model_selection import train_test_split

from src.dataset import RetinalDataset
from src.model import build_model
from src.train import train_epoch, val_epoch, get_class_weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_DIR = "../data/aptos2019-blindness-detection/train_images"

df = pd.read_csv("../data/aptos2019-blindness-detection/train.csv")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['diagnosis'])

print(f"Train: {len(train_df)} | Val: {len(val_df)}")
print("\nTrain distribution:")
print(train_df['diagnosis'].value_counts().sort_index())
print("\nVal  distribution:")
print(val_df['diagnosis'].value_counts().sort_index())

Train: 2929 | Val: 733

Train distribution:
diagnosis
0    1444
1     296
2     799
3     154
4     236
Name: count, dtype: int64

Val  distribution:
diagnosis
0    361
1     74
2    200
3     39
4     59
Name: count, dtype: int64


In [6]:
train_dataset = RetinalDataset(train_df, IMG_DIR, augment=True)
val_dataset = RetinalDataset(val_df, IMG_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 184 | Val batches: 46


In [8]:
model = build_model(num_classes=5, freeze_backbone=True).to(device)

class_weights = get_class_weights(train_df, device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

print("Model ready")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Frozen parameters: {sum(p.numel() for p in model.parameters() if not p.requires_grad):,}")

Model ready
Trainable parameters: 525,829
Frozen parameters: 23,508,032


In [9]:
EPOCHS = 10
best_kappa = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, kappa = val_epoch(model, val_loader, criterion, device)
    
    if kappa > best_kappa:
        best_kappa = kappa
        torch.save(model.state_dict(), "../outputs/best_model.pth")
        saved = "zbs, saved"
    else:
        saved = ""
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.3f} | "
          f"Kappa: {kappa:.3f} {saved}")

print(f"\nBest Kappa: {best_kappa:.3f}")

Epoch 01/10 | Train Loss: 1.374 | Train Acc: 0.540 | Val Loss: 1.202 | Val Acc: 0.666 | Kappa: 0.650 zbs, saved
Epoch 02/10 | Train Loss: 1.183 | Train Acc: 0.605 | Val Loss: 1.172 | Val Acc: 0.621 | Kappa: 0.795 zbs, saved
Epoch 03/10 | Train Loss: 1.123 | Train Acc: 0.636 | Val Loss: 1.144 | Val Acc: 0.658 | Kappa: 0.749 
Epoch 04/10 | Train Loss: 1.104 | Train Acc: 0.623 | Val Loss: 1.240 | Val Acc: 0.693 | Kappa: 0.654 
Epoch 05/10 | Train Loss: 1.114 | Train Acc: 0.637 | Val Loss: 1.202 | Val Acc: 0.738 | Kappa: 0.776 
Epoch 06/10 | Train Loss: 1.101 | Train Acc: 0.637 | Val Loss: 1.067 | Val Acc: 0.746 | Kappa: 0.788 
Epoch 07/10 | Train Loss: 1.070 | Train Acc: 0.654 | Val Loss: 1.096 | Val Acc: 0.666 | Kappa: 0.751 
Epoch 08/10 | Train Loss: 1.082 | Train Acc: 0.648 | Val Loss: 1.072 | Val Acc: 0.658 | Kappa: 0.812 zbs, saved
Epoch 09/10 | Train Loss: 1.038 | Train Acc: 0.672 | Val Loss: 1.115 | Val Acc: 0.689 | Kappa: 0.751 
Epoch 10/10 | Train Loss: 1.086 | Train Acc: 0.661 |

# Training Results — Frozen Backbone (Phase 1)

- Epochs: 10
- Best Kappa: 0.812 (Epoch 8)
- Final Train Loss: 1.086 | Final Val Loss: 1.049
- No significant overfitting observed (train acc ≈ val acc)
- Backbone frozen, only final layers trained
- Next: unfreeze backbone, fine-tune at lower learning rate

In [10]:
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

print("Backbone unfrozen")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Backbone unfrozen
Trainable parameters: 24,033,861


In [11]:
EPOCHS_FT = 5
best_kappa_ft = best_kappa

for epoch in range(EPOCHS_FT):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, kappa = val_epoch(model, val_loader, criterion, device)
    
    if kappa > best_kappa_ft:
        best_kappa_ft = kappa
        torch.save(model.state_dict(), "../outputs/best_model.pth")
        saved = "zbs saved"
    else:
        saved = ""
    
    print(f"Finetune {epoch+1:02d}/{EPOCHS_FT} | "
          f"Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.3f} | "
          f"Kappa: {kappa:.3f} {saved}")

print(f"\nBest Kappa after finetuning: {best_kappa_ft:.3f}")

Finetune 01/5 | Train Loss: 1.000 | Train Acc: 0.671 | Val Loss: 0.970 | Val Acc: 0.709 | Kappa: 0.827 zbs saved
Finetune 02/5 | Train Loss: 0.913 | Train Acc: 0.717 | Val Loss: 0.952 | Val Acc: 0.729 | Kappa: 0.834 zbs saved
Finetune 03/5 | Train Loss: 0.894 | Train Acc: 0.723 | Val Loss: 0.942 | Val Acc: 0.719 | Kappa: 0.819 
Finetune 04/5 | Train Loss: 0.857 | Train Acc: 0.738 | Val Loss: 0.902 | Val Acc: 0.746 | Kappa: 0.851 zbs saved
Finetune 05/5 | Train Loss: 0.837 | Train Acc: 0.751 | Val Loss: 0.905 | Val Acc: 0.776 | Kappa: 0.868 zbs saved

Best Kappa after finetuning: 0.868


# Fine-tuning Results — Unfrozen Backbone (Phase 2)

- Epochs: 5
- Learning rate: 1e-5 (reduced from 1e-3 to avoid catastrophic forgetting)
- Best Kappa: 0.868 (Epoch 5)
- Improvement over Phase 1: +0.056
- Training accuracy steadily improved (67% → 75%) with no overfitting signs
- Best model saved to outputs/best_model.pth